# 03 · Memoria e contesto

Di base ogni chiamata al modello è **senza memoria**. Qui vediamo come dare all'agente:
1. **memoria di breve termine** dentro una conversazione (checkpoint + `thread_id`);
2. **memoria di lungo termine** che sopravvive tra conversazioni (uno store + tool).

## Setup (autonomo)

Ogni notebook è **indipendente**: non importa nulla dal progetto. Qui carichiamo la chiave
API dal file `.env` e creiamo un modello. Esegui le celle in ordine dall'alto verso il basso.

In [ ]:
# Carichiamo le variabili d'ambiente dal file `.env`.
# Lo cerchiamo nella cartella corrente e in quelle superiori, così il notebook
# funziona sia se avviato dalla radice del progetto sia dalla cartella `notebooks`.
import os
from pathlib import Path

from dotenv import load_dotenv


def trova_env() -> Path:
    for cartella in (Path.cwd(), *Path.cwd().resolve().parents):
        if (cartella / ".env").is_file():
            return cartella / ".env"
    raise FileNotFoundError("File .env non trovato: copia .env.example in .env e aggiungi la chiave.")


env_file = trova_env()
load_dotenv(env_file, override=False)          # carica le variabili senza sovrascrivere quelle già presenti
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY mancante nel file .env"
print("Ambiente caricato da:", env_file)

In [ ]:
# `ChatOpenAI` è il wrapper LangChain attorno al modello.
# Lo creiamo una volta e lo riusiamo in tutto il notebook.
from langchain_openai import ChatOpenAI

MODELLO = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")   # modello economico, va bene per imparare
model = ChatOpenAI(
    model=MODELLO,
    use_responses_api=True,   # API "responses" di OpenAI
    store=False,              # non conservare la conversazione sui server OpenAI
)
print("Modello pronto:", MODELLO)

## 1 · Nessuna memoria (il problema)

Due chiamate separate non condividono nulla: la seconda non "ricorda" la prima.

In [ ]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Mi chiamo Nico.")])
seconda = model.invoke([HumanMessage(content="Come mi chiamo?")])
print(seconda.text)   # non può saperlo: le due chiamate sono scollegate

## 2 · Memoria di breve termine con checkpoint

Un **checkpointer** salva lo stato della conversazione. Se usiamo lo stesso `thread_id`,
l'agente ritrova i messaggi precedenti e quindi "ricorda".

In [ ]:
# InMemorySaver tiene lo stato in RAM (per imparare). In produzione: SQLite/Postgres.
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

agente = create_agent(
    model=model,
    tools=[],
    checkpointer=InMemorySaver(),   # abilita la memoria per-thread
)

In [ ]:
# `thread_id` identifica la conversazione: stesso id = stessa memoria.
config = {"configurable": {"thread_id": "conversazione-1"}}

agente.invoke({"messages": [{"role": "user", "content": "Mi chiamo Nico."}]}, config=config)
r = agente.invoke({"messages": [{"role": "user", "content": "Come mi chiamo?"}]}, config=config)
print(r["messages"][-1].text)   # ora ricorda: stesso thread_id

In [ ]:
# Cambiando thread_id ripartiamo da zero: memoria isolata per conversazione.
altro = {"configurable": {"thread_id": "conversazione-2"}}
r2 = agente.invoke({"messages": [{"role": "user", "content": "Come mi chiamo?"}]}, config=altro)
print(r2["messages"][-1].text)   # non lo sa: è un'altra conversazione

## 3 · Memoria di lungo termine con uno store + tool

La memoria di breve termine vive dentro una conversazione. Per ricordare *tra* conversazioni
serve uno **store** persistente. Lo simuliamo con un dizionario e due tool per salvare/leggere.

In [ ]:
# Uno store globale semplicissimo (un dizionario). In produzione: un DB o LangGraph Store.
from langchain_core.tools import tool

STORE: dict[str, str] = {}


@tool
def ricorda(chiave: str, valore: str) -> str:
    """Salva un'informazione durevole da ricordare in futuro."""
    STORE[chiave] = valore
    return f"Memorizzato '{chiave}'."


@tool
def richiama(chiave: str) -> str:
    """Recupera un'informazione salvata in precedenza."""
    return STORE.get(chiave, "(niente in memoria per questa chiave)")

In [ ]:
# Prima conversazione: l'agente salva una preferenza.
agente_memoria = create_agent(model=model, tools=[ricorda, richiama])
agente_memoria.invoke({"messages": [{
    "role": "user",
    "content": "Ricorda che preferisco risposte in elenco puntato. Usa il tool ricorda.",
}]})
print("Store dopo il salvataggio:", STORE)

In [ ]:
# Seconda conversazione (nuovo agente, nessuna memoria di breve termine):
# eppure recupera la preferenza dallo store di lungo termine.
agente_nuovo = create_agent(model=model, tools=[ricorda, richiama])
r = agente_nuovo.invoke({"messages": [{
    "role": "user",
    "content": "Come preferisco le risposte? Controlla la memoria con richiama.",
}]})
print(r["messages"][-1].text)

## Prova tu

- Aggiungi al system prompt: "All'inizio controlla sempre la memoria con `richiama`."
- Con conversazioni lunghe, il contesto cresce: cerca `SummarizationMiddleware` per riassumerlo.

**Idea chiave**: *breve termine* = stato della conversazione (checkpoint + thread_id);
*lungo termine* = uno store esterno che l'agente consulta con dei tool.